In [1]:
%pip install langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Clé API Groq : ")

In [3]:
from langchain.tools import tool
import outils_api as api

@tool
def rechercher_cible_chembl(nom_cible: str) -> str:
    """Cherche une protéine cible humaine dans ChEMBL à partir de son nom (ex. 'EGFR')
    et renvoie son identifiant ChEMBL (ex. CHEMBL203)."""
    return str(api.chercher_cible_chembl(nom_cible) or "Aucune cible trouvée.")

@tool
def rechercher_molecule_chembl(nom_molecule: str) -> str:
    """Cherche une molécule dans ChEMBL à partir de son nom (ex. 'gefitinib')
    et renvoie son identifiant ChEMBL."""
    return str(api.chercher_molecule_chembl(nom_molecule) or "Molécule non trouvée.")

@tool
def activites_cible(target_chembl_id: str, n: int = 10) -> str:
    """Renvoie les n molécules les plus puissantes (IC50 la plus basse) sur une cible ChEMBL,
    avec leur nom si disponible et leurs sources. Il faut un identifiant de cible comme
    CHEMBL203, obtenu avec rechercher_cible_chembl."""
    return str(api.activites_cible_chembl(target_chembl_id, n=n) or "Aucune activité trouvée.")

@tool
def fiche_molecule_pubchem(nom_molecule: str) -> str:
    """Renvoie la formule chimique, le poids moléculaire et le nom IUPAC d'une molécule
    à partir de son nom, avec son identifiant PubChem (CID)."""
    return str(api.chercher_molecule_pubchem(nom_molecule) or "Molécule non trouvée.")

@tool
def compter_activites(target_chembl_id: str) -> str:
    """Compte le nombre total de mesures d'activité enregistrées dans ChEMBL pour une cible
    (ex. CHEMBL203). Utile pour 'combien de molécules ont été testées contre la protéine X'.
    Il faut d'abord obtenir l'identifiant de la cible avec rechercher_cible_chembl."""
    return str(api.compter_activites_chembl(target_chembl_id) or "Comptage impossible.")

@tool
def molecule_par_smiles(smiles: str) -> str:
    """Cherche dans PubChem une molécule à partir de sa structure SMILES. Indique si elle est
    connue (identifiant CID), ses propriétés et le nombre d'essais biologiques où elle est active."""
    return str(api.chercher_smiles_pubchem(smiles) or "Molécule non trouvée dans PubChem.")

OUTILS = [rechercher_cible_chembl, rechercher_molecule_chembl, activites_cible,
          fiche_molecule_pubchem, compter_activites, molecule_par_smiles]

In [7]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

PROMPT_SYSTEME = """Tu es un assistant scientifique pour le criblage virtuel.
Réponds dans la langue de la question.

Règles :
1. Pour toute question factuelle sur une molécule, une cible ou une activité biologique,
   utilise OBLIGATOIREMENT les outils. N'invente jamais une valeur.
2. Commence par une phrase de réponse courte, puis donne les valeurs et les références.
3. Recopie les valeurs EXACTEMENT comme les outils les renvoient, avec leur unité.
   Ne les arrondis pas, ne les convertis pas.
4. N'ajoute AUCUNE information qui n'est pas dans les résultats des outils : pas de
   commentaire, de comparaison, d'interprétation, ni de détail supplémentaire
   (par exemple un nombre de mesures par molécule).
5. Cite pour chaque résultat les identifiants utilisés (ChEMBL, PubChem, assay, document).
6. Uniquement pour une question de comptage : le résultat est un nombre de MESURES
   d'activité enregistrées pour la cible, pas un nombre de molécules distinctes.
   Précise-le dans ta réponse.
7. Si les outils ne renvoient rien, dis-le clairement et n'essaie pas de deviner."""

agent = create_agent(model=llm, tools=OUTILS, system_prompt=PROMPT_SYSTEME)

In [5]:
import os, requests

r = requests.get("https://api.groq.com/openai/v1/models",
                 headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
                 timeout=30)
print(r.status_code)
for m in r.json()["data"]:
    print(m["id"])

200
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
allam-2-7b
qwen/qwen3.8-27b
whisper-large-v3
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m


In [8]:
def demander(question, voir_outils=True):
    resultat = agent.invoke({"messages": [{"role": "user", "content": question}]})
    if voir_outils:
        for m in resultat["messages"]:
            for appel in getattr(m, "tool_calls", None) or []:
                print("Outil appelé :", appel["name"], appel["args"])
    return resultat["messages"][-1].content

print(demander("Quelle est la formule chimique et le poids moléculaire du géfitinib ?"))
print("\n" + "-" * 60 + "\n")
print(demander("Quels sont les 5 inhibiteurs les plus puissants du récepteur EGFR ?"))
print(demander("Combien de mesures d'activité sont enregistrées dans ChEMBL pour la protéine VEGFR2 ?"))
print("\n" + "-" * 60 + "\n")
print(demander("Quels sont les 5 inhibiteurs les plus puissants du récepteur EGFR ?"))
print("\n" + "-" * 60 + "\n")
print(demander("Cette molécule est-elle connue dans PubChem ? SMILES : COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4"))

Outil appelé : fiche_molecule_pubchem {'nom_molecule': 'gefitinib'}
Le géfitinib a pour formule chimique **C22H24ClFN4O3** et un poids moléculaire de **446.9** g·mol⁻¹ (PubChem CID 123631).

------------------------------------------------------------

Outil appelé : rechercher_cible_chembl {'nom_cible': 'EGFR'}
Outil appelé : activites_cible {'n': 5, 'target_chembl_id': 'CHEMBL203'}
Les 5 inhibiteurs les plus puissants du récepteur EGFR sont :

- CHEMBL176582 : IC₅₀ = 0.01 nM (pChEMBL = 11.00) – assay CHEMBL683040 – document CHEMBL1129035  
- CHEMBL4650319 (MOBOCERTINIB) : IC₅₀ = 0.01 nM (pChEMBL = 11.00) – assay CHEMBL5260456 – document CHEMBL5257147  
- CHEMBL5270693 : IC₅₀ = 0.01 nM (pChEMBL = 11.00) – assay CHEMBL6089387 – document CHEMBL6087440  
- CHEMBL5790648 : IC₅₀ = 0.016 nM (pChEMBL = 10.80) – assay CHEMBL5736969 – document CHEMBL5728542  
- CHEMBL5746224 : IC₅₀ = 0.017 nM (pChEMBL = 10.77) – assay CHEMBL5736969 – document CHEMBL5728542
Outil appelé : rechercher_cible_chemb